# Posterior Sampling Workflow

robotorchanのsamplingはBoTorch-firstです。samplerはモデル名ではなく**posterior semantics**から選びます。

```text
candidate X
    ↓
model.posterior(X)
    ↓
posterior semantics
    ↓
BoTorch sampler
    ↓
sample_shape × batch_shape × q × output_shape
    ↓
objective / acquisition
```

このNotebookではGaussian posteriorのjoint samplingからMC acquisitionとcandidate生成まで接続します。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.objective import LinearMCObjective
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from botorch.sampling.normal import IIDNormalSampler, SobolQMCNormalSampler

from robotorchan.models import ModelListGP, SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Single-output Gaussian posterior

まず最も基本的な`SingleTaskGP`です。

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * X


train_X = torch.linspace(0.05, 0.95, 9).unsqueeze(-1)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()

candidate_X = torch.tensor([[0.25], [0.65]])
posterior = model.posterior(candidate_X)
print("posterior mean shape:", posterior.mean.shape)
print("posterior variance shape:", posterior.variance.shape)

## 3. Sobol QMC and IID normal sampling

Gaussian posteriorならBoTorch native normal samplersを直接利用できます。`sample_shape=[64]`、`q=2`、single-outputなのでsample shapeは`[64, 2, 1]`です。

In [ ]:
qmc_sampler = SobolQMCNormalSampler(torch.Size([64]), seed=123)
iid_sampler = IIDNormalSampler(torch.Size([64]), seed=123)

qmc_samples = qmc_sampler(posterior)
iid_samples = iid_sampler(posterior)

assert qmc_samples.shape == torch.Size([64, 2, 1])
assert iid_samples.shape == torch.Size([64, 2, 1])
print("QMC samples:", qmc_samples.shape)
print("IID samples:", iid_samples.shape)

## 4. q points are sampled jointly

`q=2`は2点を独立な1点posteriorとして別々にsamplingする意味ではありません。posterior covarianceを保持したままjoint sampleを生成します。

In [ ]:
sample_covariance = torch.cov(qmc_samples[..., 0].transpose(0, 1))
print("empirical covariance across q points:")
print(sample_covariance)

## 5. Multi-output ModelListGP is still a joint Gaussian path

`ModelListGP`という名前だけを見て`ListSampler`を選びません。homogeneous Gaussian submodelsではjoint multi-output Gaussian posteriorが返り、normal samplerを使えます。

In [ ]:
second_Y = torch.cos(2 * torch.pi * train_X)
multi_model = ModelListGP(
    SingleTaskGP(train_X, train_Y),
    SingleTaskGP(train_X, second_Y),
)
fit_gpytorch_mll(multi_model.make_mll())
multi_model.eval()

multi_posterior = multi_model.posterior(candidate_X)
multi_samples = SobolQMCNormalSampler(
    torch.Size([32]),
    seed=456,
)(multi_posterior)

assert multi_samples.shape == torch.Size([32, 2, 2])
print("multi-output posterior mean:", multi_posterior.mean.shape)
print("joint samples:", multi_samples.shape)

## 6. Sampling does not define optimization utility

samplerはposterior uncertaintyをsampleするだけです。multi-output samplesをどうdecision utilityへ変換するかはObjectiveの責務です。

In [ ]:
weights = torch.tensor([0.7, 0.3])
objective = LinearMCObjective(weights=weights)
utility_samples = objective(multi_samples)

assert utility_samples.shape == torch.Size([32, 2])
print("raw multi-output samples:", multi_samples.shape)
print("scalar utility samples:", utility_samples.shape)

## 7. Sampler → Objective → MC acquisition → candidate

sampling contractをそのままqLogEIへ接続し、実際にcandidateを生成します。

In [ ]:
train_multi_Y = torch.cat([train_Y, second_Y], dim=-1)
best_f = (train_multi_Y @ weights).max()
acquisition = qLogExpectedImprovement(
    model=multi_model,
    best_f=best_f,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=789),
    objective=objective,
)
bounds = torch.tensor([[0.0], [1.0]])
candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert candidate.shape == torch.Size([2, 1])
print("joint q=2 candidate:", candidate.squeeze(-1))
print("qLogEI value:", value)

## 8. Candidate gradients are a separate capability

samplerがposteriorをsampleできることと、candidate `X`に対してgradient-based acquisition optimizationできることは別contractです。Gaussian GP pathでは現在のtested contractとしてgradientが保存されます。

In [ ]:
gradient_X = torch.tensor([[[0.42]]], requires_grad=True)
gradient_value = acquisition(gradient_X)
gradient = torch.autograd.grad(gradient_value.sum(), gradient_X)[0]

assert torch.isfinite(gradient).all()
print("candidate gradient:", gradient)

## 9. Thompson sampling is candidate generation, not MC integration

finite poolからposterior drawに基づいてcandidateを選ぶThompson samplingは、acquisition expectationを近似するMC samplerとは別の役割です。robotorchanでは`select_thompson_candidates`がBoTorch `MaxPosteriorSampling`へ委譲します。

In [ ]:
from robotorchan.acquisition import select_thompson_candidates

choices = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
selected = select_thompson_candidates(
    model=model,
    choices=choices,
    num_samples=3,
    replacement=False,
)

assert selected.shape == torch.Size([3, 1])
print("Thompson-selected candidates:", selected.squeeze(-1))

## 10. Posterior-to-sampler map

| Posterior semantics | Preferred sampler | Typical path |
| --- | --- | --- |
| Gaussian | `SobolQMCNormalSampler` | GP, MultiTask, Kronecker, mixed, MF |
| Gaussian | `IIDNormalSampler` | ordinary IID MC when desired |
| empirical ensemble | `IndexSampler` | RF / ExtraTrees style posterior |
| stochastic trajectory | `StochasticSampler` | DeepGP family |
| actual `PosteriorList` | `ListSampler` | only when posterior really is a list |

`make_model_sampler`はcapability metadataに基づくintegration helperですが、direct BoTorch sampler利用を置き換えるpublic abstractionではありません。

## 11. Sampling axes and uncertainty axes

```text
candidate X
  ↓
input perturbation scenarios       ← candidate uncertainty layer
  ↓
posterior
  ↓
MC posterior samples               ← model uncertainty layer
  ↓
objective
  ↓
risk aggregation over scenarios    ← decision-risk layer
```

scenario axisとposterior MC sample axisを同じものとして扱いません。

## 12. Important boundaries

- model family名ではなくposterior semanticsでsamplerを選びます。
- `ModelListGP`だから`ListSampler`とは限りません。
- multi-output / MultiTask / Kronecker posteriorは利用可能なjoint covarianceを分解しません。
- sampling compatibilityはgradient optimization compatibilityを保証しません。
- tree ensembleや外部estimatorはsample可能でもNumPy境界によりcandidate gradientを持たない場合があります。
- Thompson samplingはfinite-pool candidate generationで、MC acquisition integrationとは別です。
- input perturbation scenario生成はposterior samplerの責務ではありません。
- 現在の監査済みmodel setにはrobotorchan独自samplerを追加する必要はありません。

## 13. Related documentation

- `docs/optimization/posterior-sampling.md`
- `docs/development/objective-posterior-transform-audit.md`
- `29_objective_posterior_transform.ipynb`

Sampling layerの役割はposterior uncertaintyを正しいjoint semanticsでdrawすることです。candidateの価値やriskをsampling layer自身に持ち込みません。